In [1]:

#################################################################
###  This Notebook cannot run, only for showing training logs ###
#################################################################


# -*- coding: utf-8 -*-
"""
Created on Mon Jun 30 22:06:32 2025

@author: jiaxx213
"""
import torch
import torch.nn as nn
import torch.optim as optim
import pickle
import os
import sys
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split, Subset

from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn import svm
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier


from functions_basic import var_list, load_sourcedata, exist_split, create_split, random_new_seed, get_all_seeds
from functions_data import Data_Manager, Gamma_Coeff
from functions_LITT import Trainer_LITT
from functions_Events import Event_Analyzer

#method_name = "T2" # 
split_rate = 0.7
hidden_size = 16

n_batch = 32
n_epochs = 35
n_epoches_gamma = 35

#-------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print("GPU is available!")
    print(torch.cuda.get_device_name(0))
else:
    print("Warning! No GPU available, back to CPU")

var_dict = {s: i for i, s in enumerate(var_list)}
dim_x = len(var_list)

#-------------------
loaded_data = load_sourcedata()
sequences = loaded_data['sequences']
time_sequences = loaded_data['time_sequences']
targets_dict = loaded_data['targets']
last_days = loaded_data['last_days']
pid_list = loaded_data['pid_list']

#'HF','IS','ARR','CA'
oc_names = ['HF','IS','ARR','AM','CA']

oc_targets = {k: targets_dict[k] for k in oc_names}
last_days = torch.tensor(last_days, dtype=torch.float32)

# ---------------------
data_manager = Data_Manager(sequences, time_sequences, oc_targets, hidden_size, n_batch, device)
data_manager.load_full()

trainer_LITT = Trainer_LITT(data_manager)
trainer_LITT.loading()

dataloader_LITT, _ = data_manager.get_DataLoader('LITT')

trainer_LITT.run_training(n_epochs, dataloader_LITT)

GPU is available!
NVIDIA A100 80GB PCIe
Data Manager: full training loaded.
Data Manager: 835 / 3276 label flipped.

Model LITT_Trainer: input_size = 35, hidden_size = 16, outcome_size = 5
Loading checkpoint from ./BC_LITT/full_LITT_checkpoint.pkl ...
== Loaded model_LITT: epoch 27, train_loss 0.6237 ==
LITT Training Skipped! n_epochs = 35 is smaller than max_epoch+1 = 36


In [3]:
flip_dataloader = data_manager.get_Flip_DataLoader(False) # False for training, True for eval

trainer_LITT.run_flipping(n_epochs, flip_dataloader)
# gamma_state = trainer_LITT.flip_gamma_coeff.get_gamma_state()
# trainer_LITT.flip_gamma_coeff.load_gamma_state(gamma_state)

== Start Flip Training ==
LITT Epoch 0/34:  Loss (avg) = 0.0985  mask 0.0774  + result 0.0211 days 71.03  gamma changed: W = 99.9%, b = 99.9% 
LITT Epoch 1/34:  Loss (avg) = 0.0748  mask 0.0592  + result 0.0156 days 61.08  gamma changed: W = 99.9%, b = 99.9% 
LITT Epoch 2/34:  Loss (avg) = 0.0623  mask 0.0496  + result 0.0127 days 55.09  gamma changed: W = 99.9%, b = 99.9% 
LITT Epoch 3/34:  Loss (avg) = 0.0596  mask 0.0474  + result 0.0121 days 53.85  gamma changed: W = 99.9%, b = 99.9% 
LITT Epoch 4/34:  Loss (avg) = 0.0549  mask 0.0437  + result 0.0112 days 51.64  gamma changed: W = 99.9%, b = 99.9% 
LITT Epoch 5/34:  Loss (avg) = 0.0526  mask 0.0419  + result 0.0106 days 50.44  gamma changed: W = 99.9%, b = 99.9% 
LITT Epoch 6/34:  Loss (avg) = 0.0510  mask 0.0409  + result 0.0101 days 49.17  gamma changed: W = 99.9%, b = 99.9% 
LITT Epoch 7/34:  Loss (avg) = 0.0500  mask 0.0400  + result 0.0100 days 49.00  gamma changed: W = 99.9%, b = 99.9% 
LITT Epoch 8/34:  Loss (avg) = 0.0489 

In [35]:
gamma_dict = trainer_LITT.flip_gamma_coeff.state_dict()
gamma_keys = list(gamma_dict.keys())
dict_half_len = int(len(gamma_dict)/2)
W_keys = gamma_keys[:dict_half_len]
b_keys = gamma_keys[-dict_half_len:]

w_keys = [k for k in gamma_keys if k[0]=='W']
print(w_keys[-1])

print(len(trainer_LITT.flip_gamma_coeff.W_params))
print(len(trainer_LITT.flip_gamma_coeff.b_params))

from functions_data import Data_Manager, Gamma_Coeff
new_flip_gamma_coeff = Gamma_Coeff(data_manager.flip_lengths, data_manager.hidden_size)
print(len(new_flip_gamma_coeff.W_params))
print(len(new_flip_gamma_coeff.b_params))

for packed_coeff, packed_X, last_day_rates, seq_idxs, target_values, target_masks in train_dataloader:

W_params.1669
1670
835
835
835


835

In [3]:
event_analyzer = Event_Analyzer(data_manager, pid_list)
event_analyzer.load_events()

recur_dict = event_analyzer.do_analyze()

# slt_evts = ['metformin_1', 'insulin_1', 'sulfonylurea_1', 'glp1_recpt_1', 'sglt2_inhib_1', 'glp1_recpt_1', 'ezetimibe_1', 'fibrates_1', 'bile_acid_1', 'diuretics_1', 'diabetes_1', 'hypertension_1', 'obesity_1', 'smoking_1', 'ckd_1']
# event_analyzer.disp_3heatmap([("radiation",1)], [("anthracyclins",1)], [("targeted",1)], slt_evts)

Loading existing timing from ./BC_LITT/full_LITT_timing.pkl ...
Recurr Round 0:
 kurt_arr is [37.15619368 36.0349282  35.47434439 17.84155545 17.55822453 16.91335383
 15.41143056 15.18894462 14.48549464 13.57459322 11.96878965  5.75940361
  4.80403055  4.09134946  3.10589024] ---- 0:3 chosen
    event_eval_df is:
           radiation_1  targeted_1  anthracyclins_1
kurt         37.156194   36.034928        35.474344
size       2281.000000  830.000000       850.000000
body_rate     0.973000    0.970000         0.973000
-----------------------------------
Recurr Round 1:
    cur_rout = [('radiation', 1)]
    cur_cond_events = {'anthracyclins': 0, 'targeted': 0, 'radiation': 1, 'metformin': 0, 'insulin': 0, 'sulfonylurea': 0, 'sglt2_inhib': 0, 'glp1_recpt': 0, 'ezetimibe': 0, 'pcsk9_inhib': 0, 'fibrates': 0, 'bile_acid': 0, 'niacin': 0, 'diuretics': 0, 'hyperlipidemia': 0, 'diabetes': 0, 'hypertension': 0, 'obesity': 0, 'smoking': 0, 'ckd': 0}
 kurt_arr is [41.20674485 31.99123873 30.06294

/scratch/ahcie-gpu2/ZhangR-Req00721/jiaxx213/LITT_reconstruct/functions_Events.py:531: RuntimeWarning: invalid value encountered in divide
  z_scores = (kurt_diff_rel - mean) / std
/scratch/ahcie-gpu2/ZhangR-Req00721/jiaxx213/LITT_reconstruct/functions_Events.py:531: RuntimeWarning: invalid value encountered in divide
  z_scores = (kurt_diff_rel - mean) / std
/scratch/ahcie-gpu2/ZhangR-Req00721/jiaxx213/LITT_reconstruct/functions_Events.py:531: RuntimeWarning: invalid value encountered in divide
  z_scores = (kurt_diff_rel - mean) / std
/scratch/ahcie-gpu2/ZhangR-Req00721/jiaxx213/LITT_reconstruct/functions_Events.py:531: RuntimeWarning: invalid value encountered in divide
  z_scores = (kurt_diff_rel - mean) / std
/scratch/ahcie-gpu2/ZhangR-Req00721/jiaxx213/LITT_reconstruct/functions_Events.py:531: RuntimeWarning: invalid value encountered in divide
  z_scores = (kurt_diff_rel - mean) / std
/scratch/ahcie-gpu2/ZhangR-Req00721/jiaxx213/LITT_reconstruct/functions_Events.py:531: RuntimeW

 kurt_arr is [45.38155256 19.82198167  6.46448501] ---- no chosen
    event_eval_df is:
Empty DataFrame
Columns: []
Index: [kurt, size, body_rate]
-----------------------------------
Recurr Round 232:
    cur_rout = [('radiation', 1), ('radiation', 2), ('radiation', 3), ('targeted', 1), ('anthracyclins', 1), ('anthracyclins', 2)]
    cur_cond_events = {'anthracyclins': 2, 'targeted': 1, 'radiation': 3, 'metformin': 0, 'insulin': 0, 'sulfonylurea': 0, 'sglt2_inhib': 0, 'glp1_recpt': 0, 'ezetimibe': 0, 'pcsk9_inhib': 0, 'fibrates': 0, 'bile_acid': 0, 'niacin': 0, 'diuretics': 0, 'hyperlipidemia': 0, 'diabetes': 0, 'hypertension': 0, 'obesity': 0, 'smoking': 0, 'ckd': 0}
 kurt_arr is [48.46501588  5.15902581  3.82475226  3.50358581] ---- 0:1 chosen
    event_eval_df is:
           targeted_2
kurt        48.465016
size        74.000000
body_rate    0.986000
-----------------------------------
Recurr Round 233:
    cur_rout = [('radiation', 1), ('radiation', 2), ('radiation', 3), ('targeted

/home/jiaxx213/my311space/lib64/python3.11/site-packages/numpy/_core/_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/jiaxx213/my311space/lib64/python3.11/site-packages/numpy/_core/_methods.py:223: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/jiaxx213/my311space/lib64/python3.11/site-packages/numpy/_core/_methods.py:181: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/home/jiaxx213/my311space/lib64/python3.11/site-packages/numpy/_core/_methods.py:215: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/scratch/ahcie-gpu2/ZhangR-Req00721/jiaxx213/LITT_reconstruct/functions_Events.py:531: RuntimeWarning: invalid value encountered in divide
  z_scores = (kurt_diff_rel - mean) / std
/scratch/ahcie-gpu2/ZhangR-Req00721/jiaxx213/LITT_reconstruct/funct

 kurt_arr is [16.08164009 11.42877816  9.12271621  8.65076589  6.82931884  6.49242266
  4.91645233  3.92986816] ---- 0:1 chosen
    event_eval_df is:
           targeted_1
kurt         16.08164
size         22.00000
body_rate     0.95500
-----------------------------------
Recurr Round 860:
    cur_rout = [('anthracyclins', 1), ('radiation', 1), ('hypertension', 1), ('anthracyclins', 2), ('anthracyclins', 3), ('radiation', 2), ('radiation', 3), ('diabetes', 1)]
    cur_cond_events = {'anthracyclins': 3, 'targeted': 0, 'radiation': 3, 'metformin': 0, 'insulin': 0, 'sulfonylurea': 0, 'sglt2_inhib': 0, 'glp1_recpt': 0, 'ezetimibe': 0, 'pcsk9_inhib': 0, 'fibrates': 0, 'bile_acid': 0, 'niacin': 0, 'diuretics': 0, 'hyperlipidemia': 0, 'diabetes': 1, 'hypertension': 1, 'obesity': 0, 'smoking': 0, 'ckd': 0}
 kurt_arr is [13.47155566  8.41514376  4.70422146] ---- no chosen
    event_eval_df is:
Empty DataFrame
Columns: []
Index: [kurt, size, body_rate]
-----------------------------------
Recurr